# Sobel vs Canny para Detecção de Trincas em Concreto

Este notebook apresenta uma comparação prática entre dois métodos clássicos de detecção de bordas em visão computacional:

- **Operador Sobel**
- **Detector de bordas Canny**

A aplicação didática é voltada para imagens de obras e inspeção visual de concreto, com foco em **realce de bordas finas**, como trincas, fissuras, juntas, descontinuidades e contornos estruturais.

**Tema da aula:** Fluxo de um sistema baseado em visão computacional  
**Etapa:** Pré-processamento / Extração de características  
**Técnica:** Detecção de bordas com Sobel e Canny  
**Versão:** 2.0 — Google Colab + Processamento em Lote  
**Data:** Agosto de 2025

## 1. Por que detectar bordas?

Em visão computacional, bordas são regiões da imagem onde há variação brusca de intensidade.

Em imagens de concreto, uma trinca pode aparecer como uma linha escura, fina e irregular. Essa linha cria uma transição visual entre a superfície do concreto e a região da fissura.

A detecção de bordas pode ajudar a:

- realçar trincas e fissuras;
- identificar contornos de patologias;
- destacar juntas e interfaces construtivas;
- preparar imagens para segmentação;
- apoiar a geração de máscaras;
- comparar resultados de diferentes métodos de pré-processamento.

A técnica não substitui um modelo de detecção ou segmentação, mas pode ser uma etapa útil no pipeline.

## 2. Sobel e Canny: visão geral

### Sobel

O **Sobel** é um operador baseado em gradiente. Ele calcula a variação da intensidade dos pixels nas direções X e Y. Depois, essas duas respostas são combinadas para gerar a magnitude do gradiente.

Fluxo simplificado:

```text
Imagem em cinza
→ suavização
→ gradiente em X
→ gradiente em Y
→ magnitude do gradiente
→ bordas
```

O Sobel é simples, rápido e interpretável, mas pode realçar ruídos e texturas do concreto como se fossem bordas.

### Canny

O **Canny** é um detector mais completo. Ele usa suavização, cálculo de gradiente, supressão de não máximos e dois limiares para formar bordas mais finas e conectadas.

O resultado depende muito dos limiares:

```python
canny_thresh1 = 50
canny_thresh2 = 150
```

- limiar baixo demais: detecta muito ruído;
- limiar alto demais: pode perder trincas finas.

## 3. Comparação conceitual

| Critério | Sobel | Canny |
|---|---|---|
| Tipo | Gradiente simples | Detector de bordas completo |
| Saída | Intensidade do gradiente | Bordas binárias |
| Sensibilidade a ruído | Maior | Menor, se bem parametrizado |
| Controle por parâmetros | Kernel e limiar | Dois limiares e suavização |
| Vantagem | Simples e interpretável | Bordas mais finas e limpas |
| Risco | Realçar textura como borda | Perder bordas fracas se limiar alto |

Na prática, vale comparar os dois para entender qual se comporta melhor em cada tipo de imagem.

## 4. Estratégia usada neste notebook

O notebook executa o seguinte fluxo:

```text
Imagem original
→ redimensionamento opcional
→ escala de cinza
→ suavização gaussiana
→ Sobel X e Sobel Y
→ magnitude Sobel
→ binarização com Otsu
→ limpeza morfológica
→ Canny
→ overlays sobre a imagem original
→ relatório comparativo
```

Para cada imagem, o notebook salva uma subpasta com:

- imagem original redimensionada;
- Sobel X;
- Sobel Y;
- magnitude Sobel;
- bordas Sobel;
- overlay Sobel;
- bordas Canny;
- overlay Canny;
- visualização comparativa;
- relatório CSV consolidado.

## 5. Importação das bibliotecas

In [ ]:
import cv2
import matplotlib.pyplot as plt
import numpy as np
import os
from google.colab import drive
import pandas as pd
from pathlib import Path

## 6. Montagem do Google Drive e configuração das pastas

O notebook espera a seguinte estrutura no Google Drive:

```text
MyDrive/
└── Python_VC/
    ├── fotos_obra/
    └── output_sobel_canny/
```

As imagens originais devem estar dentro da pasta `fotos_obra`.

In [ ]:
# Montar o Google Drive
drive.mount('/content/drive')

# Configurações principais
PASTA_BASE = '/content/drive/MyDrive/Python_VC'
PASTA_ENTRADA = os.path.join(PASTA_BASE, "fotos_obra")
PASTA_SAIDA = os.path.join(PASTA_BASE, "output_sobel_canny")

# Extensões aceitas
EXTENSOES_IMAGEM = ('.jpg', '.jpeg', '.png', '.bmp', '.tiff', '.tif', '.webp')

# Parâmetros do algoritmo
PARAMS_SOBEL_CANNY = {
    'ksize_sobel': 3,                    # Tamanho do kernel Sobel; 3 é adequado para bordas finas
    'gauss_ksize': (3, 3),               # Suavização gaussiana antes das bordas
    'gauss_sigma': 0,                    # Sigma automático
    'canny_thresh1': 50,                 # Limiar inferior Canny
    'canny_thresh2': 150,                # Limiar superior Canny
    'redimensionar_max': 1200,           # Largura máxima para redimensionamento
    'morph_kernel_size': 3,              # Kernel para limpeza morfológica no Sobel
    'salvar_visualizacao': True          # Salvar comparativo visual por imagem
}

print("Configuração carregada.")
print(f"Pasta base: {PASTA_BASE}")
print(f"Pasta de entrada: {PASTA_ENTRADA}")
print(f"Pasta de saída: {PASTA_SAIDA}")
print(f"Parâmetros: {PARAMS_SOBEL_CANNY}")

## 7. Verificação da configuração

In [ ]:
def verificar_configuracao():
    """Verifica se todos os arquivos e pastas necessários existem."""
    print("🔍 VERIFICANDO CONFIGURAÇÃO")
    print(f"📁 Pasta base: {PASTA_BASE}")
    print(f"📂 Entrada: {PASTA_ENTRADA}")
    print(f"📂 Saída: {PASTA_SAIDA}")

    problemas = []

    if not os.path.exists(PASTA_BASE):
        problemas.append("❌ Pasta base não encontrada.")

    if not os.path.exists(PASTA_ENTRADA):
        problemas.append("❌ Pasta de entrada 'fotos_obra' não encontrada.")

    if PARAMS_SOBEL_CANNY['ksize_sobel'] not in [1, 3, 5, 7]:
        problemas.append("❌ ksize_sobel deve ser 1, 3, 5 ou 7.")

    if PARAMS_SOBEL_CANNY['canny_thresh1'] >= PARAMS_SOBEL_CANNY['canny_thresh2']:
        problemas.append("❌ canny_thresh1 deve ser menor que canny_thresh2.")

    if os.path.exists(PASTA_ENTRADA):
        imagens = [
            f for f in os.listdir(PASTA_ENTRADA)
            if f.lower().endswith(EXTENSOES_IMAGEM)
        ]

        if not imagens:
            problemas.append("❌ Nenhuma imagem encontrada na pasta 'fotos_obra'.")
        else:
            print(f"📸 Imagens encontradas: {len(imagens)}")

    if problemas:
        print("\n".join(problemas))
        return False

    print("✅ Configuração verificada com sucesso.")
    return True

verificar_configuracao()

## 8. Funções auxiliares

In [ ]:
def criar_diretorio_se_nao_existir(caminho):
    """Cria diretório se ele ainda não existir."""
    os.makedirs(caminho, exist_ok=True)


def listar_imagens(pasta):
    """Lista imagens válidas em uma pasta."""
    return [
        f for f in os.listdir(pasta)
        if f.lower().endswith(EXTENSOES_IMAGEM)
    ]


def redimensionar_imagem(imagem, largura_max):
    """Redimensiona imagem mantendo proporção."""
    h, w = imagem.shape[:2]

    if largura_max and largura_max > 0 and w > largura_max:
        ratio = largura_max / w
        nova_largura = largura_max
        nova_altura = int(h * ratio)

        imagem_redimensionada = cv2.resize(
            imagem,
            (nova_largura, nova_altura),
            interpolation=cv2.INTER_AREA
        )

        print(f"   📐 Redimensionada: {w}x{h} → {nova_largura}x{nova_altura}")

        return imagem_redimensionada

    return imagem


def salvar_resultado(caminho, imagem):
    """Salva imagem com verificação de diretório."""
    criar_diretorio_se_nao_existir(os.path.dirname(caminho))
    cv2.imwrite(caminho, imagem)

## 9. Pré-processamento da imagem

Antes da detecção de bordas, aplicamos:

1. carregamento da imagem;
2. redimensionamento opcional;
3. conversão para escala de cinza;
4. suavização gaussiana.

A suavização reduz ruídos que poderiam ser interpretados como bordas falsas.

In [ ]:
def preparar_imagem(caminho_imagem, params):
    """
    Carrega e prepara a imagem para Sobel e Canny.

    Retorno:
        img_bgr: imagem colorida redimensionada.
        cinza: imagem em escala de cinza.
        cinza_suavizado: imagem após suavização.
    """
    img_bgr = cv2.imread(caminho_imagem)

    if img_bgr is None:
        raise ValueError(f"Erro ao carregar a imagem: {caminho_imagem}")

    img_bgr = redimensionar_imagem(
        img_bgr,
        params['redimensionar_max']
    )

    cinza = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)

    if params['gauss_ksize'] != (0, 0):
        cinza_suavizado = cv2.GaussianBlur(
            cinza,
            params['gauss_ksize'],
            params['gauss_sigma']
        )
    else:
        cinza_suavizado = cinza.copy()

    return img_bgr, cinza, cinza_suavizado

## 10. Detecção de bordas com Sobel

In [ ]:
def detectar_bordas_sobel(cinza_suavizado, img_bgr, params):
    """
    Detecta bordas usando Sobel.

    Retorno:
        dicionário com Sobel X, Sobel Y, magnitude, bordas limpas e overlay.
    """
    sobel_x = cv2.Sobel(
        cinza_suavizado,
        cv2.CV_64F,
        1,
        0,
        ksize=params['ksize_sobel']
    )

    sobel_y = cv2.Sobel(
        cinza_suavizado,
        cv2.CV_64F,
        0,
        1,
        ksize=params['ksize_sobel']
    )

    magnitude = np.hypot(sobel_x, sobel_y)

    magnitude_normalizada = cv2.normalize(
        magnitude,
        None,
        0,
        255,
        cv2.NORM_MINMAX
    ).astype(np.uint8)

    _, bordas_sobel = cv2.threshold(
        magnitude_normalizada,
        0,
        255,
        cv2.THRESH_BINARY + cv2.THRESH_OTSU
    )

    kernel = cv2.getStructuringElement(
        cv2.MORPH_RECT,
        (params['morph_kernel_size'], params['morph_kernel_size'])
    )

    bordas_sobel_limpas = cv2.morphologyEx(
        bordas_sobel,
        cv2.MORPH_OPEN,
        kernel
    )

    overlay_sobel = img_bgr.copy()
    overlay_sobel[bordas_sobel_limpas > 0] = (0, 255, 255)
    overlay_sobel = cv2.addWeighted(img_bgr, 0.7, overlay_sobel, 0.3, 0)

    return {
        'sobel_x': sobel_x,
        'sobel_y': sobel_y,
        'magnitude': magnitude,
        'magnitude_normalizada': magnitude_normalizada,
        'bordas': bordas_sobel,
        'bordas_limpas': bordas_sobel_limpas,
        'overlay': overlay_sobel
    }

## 11. Detecção de bordas com Canny

In [ ]:
def detectar_bordas_canny(cinza_suavizado, img_bgr, params):
    """
    Detecta bordas usando Canny.

    Retorno:
        dicionário com bordas Canny e overlay.
    """
    bordas_canny = cv2.Canny(
        cinza_suavizado,
        params['canny_thresh1'],
        params['canny_thresh2']
    )

    overlay_canny = img_bgr.copy()
    overlay_canny[bordas_canny > 0] = (0, 0, 255)
    overlay_canny = cv2.addWeighted(img_bgr, 0.7, overlay_canny, 0.3, 0)

    return {
        'bordas': bordas_canny,
        'overlay': overlay_canny
    }

## 12. Métricas comparativas

Para comparar Sobel e Canny, o notebook calcula:

- quantidade de pixels detectados pelo Sobel;
- quantidade de pixels detectados pelo Canny;
- percentual da imagem marcado como borda;
- diferença percentual entre os métodos.

Essas métricas não indicam, sozinhas, qual método é melhor. Elas ajudam a identificar se um método está detectando bordas demais ou de menos.

In [ ]:
def calcular_estatisticas(nome_arquivo, img_bgr, bordas_sobel, bordas_canny):
    """Calcula estatísticas comparativas entre Sobel e Canny."""
    pixels_sobel = int(np.sum(bordas_sobel > 0))
    pixels_canny = int(np.sum(bordas_canny > 0))
    pixels_totais = int(bordas_sobel.size)

    percentual_sobel = (pixels_sobel / pixels_totais) * 100
    percentual_canny = (pixels_canny / pixels_totais) * 100
    diferenca_percentual = abs(percentual_sobel - percentual_canny)

    estatisticas = {
        'arquivo': nome_arquivo,
        'pixels_sobel': pixels_sobel,
        'pixels_canny': pixels_canny,
        'percentual_sobel': percentual_sobel,
        'percentual_canny': percentual_canny,
        'diferenca_percentual': diferenca_percentual,
        'largura': img_bgr.shape[1],
        'altura': img_bgr.shape[0],
        'ksize_sobel': PARAMS_SOBEL_CANNY['ksize_sobel'],
        'canny_thresh1': PARAMS_SOBEL_CANNY['canny_thresh1'],
        'canny_thresh2': PARAMS_SOBEL_CANNY['canny_thresh2']
    }

    return estatisticas

## 13. Salvamento dos resultados

In [ ]:
def salvar_resultados_sobel_canny(nome_arquivo, img_bgr, sobel, canny, pasta_saida, params):
    """Salva resultados de Sobel e Canny para uma imagem."""
    nome_base = os.path.splitext(nome_arquivo)[0]

    pasta_imagem = os.path.join(pasta_saida, nome_base)
    criar_diretorio_se_nao_existir(pasta_imagem)

    caminhos = {}

    caminhos['original'] = os.path.join(pasta_imagem, f"{nome_base}_original.png")
    caminhos['sobel_x'] = os.path.join(pasta_imagem, f"{nome_base}_sobel_x.png")
    caminhos['sobel_y'] = os.path.join(pasta_imagem, f"{nome_base}_sobel_y.png")
    caminhos['sobel_magnitude'] = os.path.join(pasta_imagem, f"{nome_base}_sobel_magnitude.png")
    caminhos['sobel_bordas'] = os.path.join(pasta_imagem, f"{nome_base}_sobel_bordas.png")
    caminhos['sobel_overlay'] = os.path.join(pasta_imagem, f"{nome_base}_sobel_overlay.png")
    caminhos['canny_bordas'] = os.path.join(pasta_imagem, f"{nome_base}_canny_bordas.png")
    caminhos['canny_overlay'] = os.path.join(pasta_imagem, f"{nome_base}_canny_overlay.png")

    salvar_resultado(caminhos['original'], img_bgr)
    salvar_resultado(caminhos['sobel_x'], cv2.convertScaleAbs(sobel['sobel_x']))
    salvar_resultado(caminhos['sobel_y'], cv2.convertScaleAbs(sobel['sobel_y']))
    salvar_resultado(caminhos['sobel_magnitude'], sobel['magnitude_normalizada'])
    salvar_resultado(caminhos['sobel_bordas'], sobel['bordas_limpas'])
    salvar_resultado(caminhos['sobel_overlay'], sobel['overlay'])
    salvar_resultado(caminhos['canny_bordas'], canny['bordas'])
    salvar_resultado(caminhos['canny_overlay'], canny['overlay'])

    if params['salvar_visualizacao']:
        caminhos['comparativo'] = os.path.join(
            pasta_imagem,
            f"{nome_base}_comparativo_sobel_canny.png"
        )

        fig, axes = plt.subplots(2, 4, figsize=(20, 10))

        axes[0, 0].imshow(cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB))
        axes[0, 0].set_title("Original")
        axes[0, 0].axis("off")

        axes[0, 1].imshow(sobel['magnitude_normalizada'], cmap="gray")
        axes[0, 1].set_title("Sobel — Magnitude")
        axes[0, 1].axis("off")

        axes[0, 2].imshow(sobel['bordas_limpas'], cmap="gray")
        axes[0, 2].set_title("Sobel — Bordas")
        axes[0, 2].axis("off")

        axes[0, 3].imshow(cv2.cvtColor(sobel['overlay'], cv2.COLOR_BGR2RGB))
        axes[0, 3].set_title("Sobel — Overlay")
        axes[0, 3].axis("off")

        axes[1, 0].imshow(cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB))
        axes[1, 0].set_title("Original")
        axes[1, 0].axis("off")

        axes[1, 1].imshow(canny['bordas'], cmap="gray")
        axes[1, 1].set_title(
            f"Canny — Bordas\nT1={params['canny_thresh1']}, T2={params['canny_thresh2']}"
        )
        axes[1, 1].axis("off")

        axes[1, 2].imshow(cv2.cvtColor(canny['overlay'], cv2.COLOR_BGR2RGB))
        axes[1, 2].set_title("Canny — Overlay")
        axes[1, 2].axis("off")

        comparacao = np.hstack([sobel['bordas_limpas'], canny['bordas']])
        axes[1, 3].imshow(comparacao, cmap="gray")
        axes[1, 3].set_title("Comparação: Sobel | Canny")
        axes[1, 3].axis("off")

        plt.suptitle(f"Análise de bordas — {nome_arquivo}", fontsize=16)
        plt.tight_layout()
        plt.savefig(caminhos['comparativo'], dpi=150, bbox_inches="tight")
        plt.close()

    return caminhos

## 14. Processamento de uma imagem

In [ ]:
def processar_imagem_sobel_canny(caminho_imagem, pasta_saida, params):
    """Processa uma imagem com Sobel e Canny para detecção de trincas."""
    nome_arquivo = os.path.basename(caminho_imagem)

    print(f"🔍 Processando: {nome_arquivo}")

    img_bgr, cinza, cinza_suavizado = preparar_imagem(caminho_imagem, params)

    sobel = detectar_bordas_sobel(
        cinza_suavizado,
        img_bgr,
        params
    )

    canny = detectar_bordas_canny(
        cinza_suavizado,
        img_bgr,
        params
    )

    estatisticas = calcular_estatisticas(
        nome_arquivo,
        img_bgr,
        sobel['bordas_limpas'],
        canny['bordas']
    )

    caminhos = salvar_resultados_sobel_canny(
        nome_arquivo,
        img_bgr,
        sobel,
        canny,
        pasta_saida,
        params
    )

    estatisticas.update({
        'caminho_original': caminhos.get('original'),
        'caminho_sobel_overlay': caminhos.get('sobel_overlay'),
        'caminho_canny_overlay': caminhos.get('canny_overlay'),
        'caminho_comparativo': caminhos.get('comparativo')
    })

    print(
        f"✅ Concluído: "
        f"{estatisticas['pixels_sobel']} px Sobel, "
        f"{estatisticas['pixels_canny']} px Canny"
    )

    return estatisticas

## 15. Visualização rápida de uma imagem

In [ ]:
def visualizar_resultados_comparativos(caminho_imagem, params):
    """Gera visualização comparativa para uma imagem."""
    nome_arquivo = os.path.basename(caminho_imagem)

    img_bgr, cinza, cinza_suavizado = preparar_imagem(
        caminho_imagem,
        params
    )

    sobel = detectar_bordas_sobel(
        cinza_suavizado,
        img_bgr,
        params
    )

    canny = detectar_bordas_canny(
        cinza_suavizado,
        img_bgr,
        params
    )

    fig, axes = plt.subplots(2, 4, figsize=(20, 10))

    axes[0, 0].imshow(cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB))
    axes[0, 0].set_title("Original")
    axes[0, 0].axis("off")

    axes[0, 1].imshow(sobel['magnitude_normalizada'], cmap="gray")
    axes[0, 1].set_title("Sobel — Magnitude")
    axes[0, 1].axis("off")

    axes[0, 2].imshow(sobel['bordas_limpas'], cmap="gray")
    axes[0, 2].set_title("Sobel — Bordas")
    axes[0, 2].axis("off")

    axes[0, 3].imshow(cv2.cvtColor(sobel['overlay'], cv2.COLOR_BGR2RGB))
    axes[0, 3].set_title("Sobel — Overlay")
    axes[0, 3].axis("off")

    axes[1, 0].imshow(cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB))
    axes[1, 0].set_title("Original")
    axes[1, 0].axis("off")

    axes[1, 1].imshow(canny['bordas'], cmap="gray")
    axes[1, 1].set_title(
        f"Canny — Bordas\nT1={params['canny_thresh1']}, T2={params['canny_thresh2']}"
    )
    axes[1, 1].axis("off")

    axes[1, 2].imshow(cv2.cvtColor(canny['overlay'], cv2.COLOR_BGR2RGB))
    axes[1, 2].set_title("Canny — Overlay")
    axes[1, 2].axis("off")

    comparacao = np.hstack([sobel['bordas_limpas'], canny['bordas']])
    axes[1, 3].imshow(comparacao, cmap="gray")
    axes[1, 3].set_title("Comparação: Sobel | Canny")
    axes[1, 3].axis("off")

    plt.suptitle(f"Análise de Trincas — {nome_arquivo}", fontsize=16)
    plt.tight_layout()
    plt.show()

## 16. Processamento em lote

In [ ]:
def executar_analise_sobel_canny(mostrar_visualizacoes=True):
    """Executa análise Sobel vs Canny em lote."""

    print("=== DETECÇÃO DE TRINCAS — SOBEL vs CANNY ===")
    print(f"📁 Pasta base: {PASTA_BASE}")
    print(f"📂 Entrada: {PASTA_ENTRADA}")
    print(f"📂 Saída: {PASTA_SAIDA}")
    print(f"⚙️ Parâmetros: {PARAMS_SOBEL_CANNY}")
    print("-" * 60)

    if not verificar_configuracao():
        print("\n❌ Não foi possível iniciar o processamento.")
        return None

    criar_diretorio_se_nao_existir(PASTA_SAIDA)

    arquivos_imagem = listar_imagens(PASTA_ENTRADA)

    print(f"📊 Encontradas {len(arquivos_imagem)} imagens para processamento")

    todas_estatisticas = []
    erros = 0

    for i, arquivo_imagem in enumerate(arquivos_imagem, 1):
        caminho_imagem = os.path.join(PASTA_ENTRADA, arquivo_imagem)

        print(f"\n[{i}/{len(arquivos_imagem)}] Processando: {arquivo_imagem}")

        try:
            if mostrar_visualizacoes:
                visualizar_resultados_comparativos(
                    caminho_imagem,
                    PARAMS_SOBEL_CANNY
                )

            estatisticas = processar_imagem_sobel_canny(
                caminho_imagem,
                PASTA_SAIDA,
                PARAMS_SOBEL_CANNY
            )

            if estatisticas:
                todas_estatisticas.append(estatisticas)

        except Exception as e:
            print(f"❌ Erro ao processar {arquivo_imagem}: {str(e)}")
            erros += 1

    if todas_estatisticas:
        df_estatisticas = pd.DataFrame(todas_estatisticas)

        caminho_relatorio = os.path.join(
            PASTA_SAIDA,
            "relatorio_estatisticas_sobel_canny.csv"
        )

        df_estatisticas.to_csv(caminho_relatorio, index=False)

        print("\n📈 RELATÓRIO FINAL")
        print(f"📊 Total de imagens processadas: {len(todas_estatisticas)}")
        print(f"❌ Erros: {erros}")
        print(f"📋 Média Sobel: {df_estatisticas['percentual_sobel'].mean():.2f}%")
        print(f"📋 Média Canny: {df_estatisticas['percentual_canny'].mean():.2f}%")
        print(f"📋 Diferença média: {df_estatisticas['diferenca_percentual'].mean():.2f}%")
        print(f"📁 Relatório salvo em: {caminho_relatorio}")
        print(f"🎯 Processamento concluído! Resultados em: {PASTA_SAIDA}")

        return df_estatisticas

    print("\n⚠️ Nenhuma imagem foi processada com sucesso.")
    return None

## 17. Execução completa

Execute esta célula para rodar o processamento em lote.

Para processamentos grandes, recomenda-se usar:

```python
mostrar_visualizacoes=False
```

Isso acelera a execução porque evita exibir gráficos para cada imagem.

In [ ]:
df_sobel_canny = executar_analise_sobel_canny(mostrar_visualizacoes=True)

## 18. Visualizar relatório consolidado

In [ ]:
if 'df_sobel_canny' in globals() and isinstance(df_sobel_canny, pd.DataFrame):
    display(df_sobel_canny)
else:
    print("Execute primeiro a célula de processamento completo.")

## 19. Como interpretar os resultados

A comparação deve ser feita visualmente e com apoio das métricas.

### Quando o Sobel detecta bordas demais

Pode indicar que a textura do concreto, manchas ou ruídos estão sendo realçados como bordas.

Possíveis ajustes:

```python
PARAMS_SOBEL_CANNY['morph_kernel_size'] = 5
PARAMS_SOBEL_CANNY['gauss_ksize'] = (5, 5)
```

### Quando o Canny detecta bordas demais

Os limiares podem estar baixos.

```python
PARAMS_SOBEL_CANNY['canny_thresh1'] = 80
PARAMS_SOBEL_CANNY['canny_thresh2'] = 200
```

### Quando o Canny perde trincas finas

Os limiares podem estar altos.

```python
PARAMS_SOBEL_CANNY['canny_thresh1'] = 30
PARAMS_SOBEL_CANNY['canny_thresh2'] = 100
```

### Quando a imagem tem muito ruído

Aumente a suavização.

```python
PARAMS_SOBEL_CANNY['gauss_ksize'] = (5, 5)
```

In [ ]:
# Célula opcional para ajuste de parâmetros antes de reexecutar

# Exemplo 1: Canny mais sensível para trincas finas
# PARAMS_SOBEL_CANNY['canny_thresh1'] = 30
# PARAMS_SOBEL_CANNY['canny_thresh2'] = 100

# Exemplo 2: Canny mais conservador
# PARAMS_SOBEL_CANNY['canny_thresh1'] = 80
# PARAMS_SOBEL_CANNY['canny_thresh2'] = 200

# Exemplo 3: Sobel com mais suavização
# PARAMS_SOBEL_CANNY['gauss_ksize'] = (5, 5)

# Exemplo 4: Sobel com limpeza morfológica mais forte
# PARAMS_SOBEL_CANNY['morph_kernel_size'] = 5

PARAMS_SOBEL_CANNY

## 20. Teste rápido com a primeira imagem

Esta célula permite visualizar rapidamente o resultado em apenas uma imagem, útil para calibrar os parâmetros antes de processar o lote completo.

In [ ]:
imagens_disponiveis = listar_imagens(PASTA_ENTRADA) if os.path.exists(PASTA_ENTRADA) else []

if imagens_disponiveis:
    primeira_imagem = os.path.join(PASTA_ENTRADA, imagens_disponiveis[0])
    visualizar_resultados_comparativos(primeira_imagem, PARAMS_SOBEL_CANNY)
else:
    print("Nenhuma imagem disponível para teste.")

## 21. Interpretação didática para inspeção de trincas

Em imagens de concreto:

- **Sobel** tende a destacar mudanças de textura e bordas mais amplas;
- **Canny** tende a entregar bordas mais finas e binárias;
- trincas muito fracas podem não aparecer em Canny se os limiares forem altos;
- texturas muito rugosas podem gerar muitos falsos positivos em Sobel;
- nenhum dos dois métodos identifica semanticamente uma trinca.

Portanto, o resultado deve ser entendido como **realce de bordas**, não como diagnóstico automático da patologia.

## 22. Fluxo recomendado em um pipeline de visão computacional

Um fluxo possível para análise de trincas em concreto seria:

```text
Imagem da obra
→ redimensionamento
→ correção de iluminação
→ remoção de ruído
→ CLAHE
→ Sobel/Canny para realce de bordas
→ YOLO para detecção
→ U-Net para segmentação
→ validação técnica
→ relatório de inspeção
```

Sobel e Canny podem ser usados tanto para análise exploratória quanto para gerar insumos auxiliares para outros modelos.